# 01 — Data Understanding and Exploration


## 1. Study Context and Multiclass Objective

Dry bean varieties can be distinguished using measurable morphological characteristics extracted from images of individual grains. The **Dry Bean** dataset from the UCI Machine Learning Repository provides **13,611 observations**, described by **16 numerical shape and dimensional features**, with each grain assigned to one of **seven registered bean varieties**.

This study uses those tabular measurements to investigate how the observed morphology differs across bean varieties and to establish evidence for the later development and comparison of **multiclass classification models**.

The exploratory stage will focus on understanding the dataset before any modeling decisions are made. In particular, it will examine data quality, class balance, numerical distributions, relationships and redundancies among morphological measurements, and the degree of separation or overlap between classes.

**Experimental boundary.** The notebook is organized in two parts so that the final-test holdout never informs development decisions:

- **Part A — source acquisition and structural validation (sections 2–9)** runs on the complete released table. These are row-wise contract checks (schema, declared target classes, domain rules, missing/invalid values, exact-row structure) with no learned parameters; they cannot rank features, compare classes, or select models.
- **Section 9b freezes the stratified train/validation/test split** from structural facts only (static snapshot, nominal seven-class target, no source identifier) and fingerprints its membership.
- **Part B — development exploration (sections 10–19)** runs on the **training partition only**. Every distribution-aware or target-aware analysis — class balance, numerical distributions, correlations and redundancy, feature-to-target association, class profiles/PCA, and derived-dependency confirmation — and every decision derived from them is therefore independent of the validation and final-test partitions.

The scope of this project begins with the features already extracted by the dataset authors. It does **not** attempt to reproduce the original image-acquisition, segmentation, or feature-extraction pipeline, and exploratory associations should not be interpreted as causal or agronomic effects.

The study is guided by the following questions:

- Are the seven target classes sufficiently represented, or is class imbalance likely to influence modeling and evaluation?
- Which morphological measurements most clearly differentiate the observed bean varieties?
- Which features are strongly related or mathematically redundant, and how should those relationships inform later preparation and modeling?
- Where do class profiles overlap, suggesting potentially difficult multiclass distinctions?
- Which exploratory findings should guide data preparation, feature treatment, model selection, and multiclass evaluation?

## 2. Dataset Source and Acquisition


In [ ]:
import pandas as pd

from scripts.download_data import acquire_uci_dataset
from scripts.project_context import get_project_context


# Dataset-specific source choices remain visible in the notebook.
DATASET_SLUG = "dry-bean"
UCI_DATASET_ID = 602

PROJECT = get_project_context()
RAW_DATA_DIR = PROJECT.path("data", "raw", DATASET_SLUG)

acquisition = acquire_uci_dataset(
    dataset_id=UCI_DATASET_ID,
    destination=RAW_DATA_DIR,
    project_root=PROJECT.root,
)

DATASET_FILE = acquisition.require_one_file("dataset.csv")
df = pd.read_csv(DATASET_FILE)

print(f"Dataset source: UCI ML Repository — ID {UCI_DATASET_ID}")
print(f"Raw data directory: {PROJECT.display(RAW_DATA_DIR)}")
print(f"Dataset file: {DATASET_FILE.name}")
print(f"Loaded shape: {df.shape}")

## 3. Unit of Observation


In [ ]:
from scripts.validate_data import analyze_observation_unit


# Dataset-specific analytical definition remains visible.
OBSERVATION_LABEL = "individual dry bean grain"
OBSERVATION_IDENTIFIER = None

observation_report = analyze_observation_unit(
    dataframe=df,
    identifier=OBSERVATION_IDENTIFIER,
)

print(f"Unit of observation: {OBSERVATION_LABEL}")
observation_report.summary_frame()

## 4. Dataset Structure and Source Identity


In [ ]:
from scripts.describe_dataset import describe_dataset_structure


structure_report = describe_dataset_structure(
    df,
    acquisition=acquisition,
    expected_source_id=UCI_DATASET_ID,
)

display(structure_report.summary_frame())
structure_report.columns_frame()

## 5. Target and Multiclass Prediction Contract


In [ ]:
from IPython.display import display

from scripts.target_contract import define_classification_target_contract


# Dataset-specific prediction semantics remain explicit in the notebook.
TARGET_COLUMN = "Class"
PROBLEM_TYPE = "multiclass_classification"
EXPECTED_TARGET_CLASSES = (
    "SEKER",
    "BARBUNYA",
    "BOMBAY",
    "CALI",
    "DERMASON",
    "HOROZ",
    "SIRA",
)

target_contract = define_classification_target_contract(
    df,
    target=TARGET_COLUMN,
    expected_classes=EXPECTED_TARGET_CLASSES,
    problem_type=PROBLEM_TYPE,
    source_variables_file=acquisition.require_one_file("variables.csv"),
)

display(target_contract.summary_frame())
target_contract.classes_frame()

## 6. Feature Roles and Data Dictionary


In [ ]:
from scripts.feature_dictionary import define_feature_roles_and_dictionary


feature_dictionary = define_feature_roles_and_dictionary(
    df,
    source_variables_file=acquisition.require_one_file("variables.csv"),
    target=target_contract.target,
    identifiers=OBSERVATION_IDENTIFIER,
)

FEATURE_COLUMNS = feature_dictionary.feature_columns
IDENTIFIER_COLUMNS = feature_dictionary.identifier_columns

display(feature_dictionary.summary_frame())
feature_dictionary.column_frame()

## 7. Data Types, Ranges, and Domain Validity


In [ ]:
from scripts.validate_domains import analyze_types_ranges_and_domains


# Dataset-specific physical and mathematical constraints remain visible.
DOMAIN_RULES = {
    "positive": (
        "Area",
        "Perimeter",
        "MajorAxisLength",
        "MinorAxisLength",
        "ConvexArea",
        "EquivDiameter",
        "ShapeFactor1",
        "ShapeFactor2",
        "ShapeFactor3",
        "ShapeFactor4",
    ),
    "unit_interval": (
        "Eccentricity",
        "Extent",
        "Solidity",
        "Roundness",
        "Compactness",
    ),
    "minimums": {
        "AspectRatio": 1.0,
    },
    "relations": (
        ("MajorAxisLength", ">=", "MinorAxisLength"),
        ("ConvexArea", ">=", "Area"),
    ),
}

domain_report = analyze_types_ranges_and_domains(
    df,
    source_variables_file=acquisition.require_one_file("variables.csv"),
    domain_rules=DOMAIN_RULES,
)

display(domain_report.summary_frame())
display(domain_report.column_frame())
domain_report.relation_frame()

## 8. Missing and Invalid Values


In [ ]:
from scripts.validate_values import (
    analyze_source_backed_missing_and_invalid_values,
)


value_quality_report = analyze_source_backed_missing_and_invalid_values(
    df,
    source_variables_file=acquisition.require_one_file("variables.csv"),
    target=target_contract.target,
    expected_target_values=target_contract.expected_classes,
    max_issue_samples=10,
)

# Validate complete assessment coverage while preserving any findings as
# exploratory evidence for the later preparation stage.
value_quality_report.raise_if_invalid(
    require_all_columns_assessed=True,
    require_rule_columns_present=True,
    require_no_quality_issues=False,
)

display(value_quality_report.summary_frame())
display(value_quality_report.column_frame())

if value_quality_report.has_issues:
    display(value_quality_report.issues_with_impacts_frame())
else:
    print("No missing, blank, inconsistent, or invalid values were found.")

## 9. Duplicate Records

Exact-row structure is audited on the complete source because any deduplication decision would have to precede the split. The decision rule is structural: without a source identifier, equality alone never justifies dropping a row.


In [ ]:
from scripts.validate_duplicates import analyze_duplicate_records


duplicate_report = analyze_duplicate_records(
    df,
    identifiers=IDENTIFIER_COLUMNS,
    feature_columns=FEATURE_COLUMNS,
    target=target_contract.target,
    max_group_samples=10,
)

display(duplicate_report.summary_frame())

if not duplicate_report.has_source_identifiers:
    print(
        "Source identifiers are not available. Exact row matches are "
        "review evidence, not proof that the same grain was recorded twice."
    )

if duplicate_report.has_exact_duplicates:
    print("Sample of exact row matches:")
    display(duplicate_report.exact_duplicates_frame())

if duplicate_report.has_repeated_profiles:
    print("Sample of repeated candidate-feature profiles:")
    display(duplicate_report.repeated_profiles_frame())

if duplicate_report.has_target_conflicts:
    print(
        "Some identical feature profiles map to different classes. "
        "Treat these cases as analytical ambiguity rather than automatic "
        "data-quality errors."
    )
    display(duplicate_report.target_conflicts_frame())

# Without source identifiers, equality alone does not justify dropping rows.
duplicate_report.raise_if_invalid(
    require_no_exact_duplicates=bool(IDENTIFIER_COLUMNS),
    require_unique_identifiers=bool(IDENTIFIER_COLUMNS),
    require_no_conflicting_identifiers=bool(IDENTIFIER_COLUMNS),
    require_no_repeated_profiles=False,
    require_no_target_conflicts=False,
)

## 9b. Development Split Freeze Before Distribution-Aware Exploration

The split policy is declared from structural facts only: the released table is a static snapshot without a chronological field, the target is nominal with seven declared classes, and no source identifier exists. The stratified 70/15/15 split with seed 42 is executed here, before any distribution-aware or target-aware analysis, and its partition and membership fingerprints are recorded in the exploration handoff. Notebook 02 must reproduce exactly this membership.

From this point onward only the **training partition** (`development_df`) is analysed. Validation and test rows are discarded from this notebook's memory and are never inspected here.


In [ ]:
from scripts.prepare_data import (
    ClassificationSplitPolicy,
    build_development_split_reference,
    split_classification_dataset,
)


# Split policy is declared from structural facts only, before any
# distribution-aware or target-aware exploration.
TRAIN_FRACTION = 0.70
VALIDATION_FRACTION = 0.15
TEST_FRACTION = 0.15
PREPARATION_RANDOM_SEED = 42
EDA_PARTITION = "train"

DEVELOPMENT_SPLIT_POLICY = ClassificationSplitPolicy(
    evaluation_mode="stratified_random_snapshot",
    purpose="educational_benchmark",
    train_fraction=TRAIN_FRACTION,
    validation_fraction=VALIDATION_FRACTION,
    test_fraction=TEST_FRACTION,
    stratify_by=target_contract.target,
    random_seed=PREPARATION_RANDOM_SEED,
    shuffle=True,
    educational_justification=(
        "Use the source-released static Dry Bean snapshot for a reproducible "
        "educational multiclass benchmark without claiming future or "
        "production validity."
    ),
    operational_validity="unconfirmed",
    temporal_contract_status="resolved_static_snapshot",
    feature_inference_availability="unconfirmed",
)

development_partitions = split_classification_dataset(
    df,
    policy=DEVELOPMENT_SPLIT_POLICY,
    identifier_columns=IDENTIFIER_COLUMNS,
    target_classes=target_contract.expected_classes,
)
DEVELOPMENT_SPLIT = build_development_split_reference(
    development_partitions,
    policy=DEVELOPMENT_SPLIT_POLICY,
    eda_partition=EDA_PARTITION,
)
development_df = development_partitions.train
# Validation and test rows are not retained by this notebook.
del development_partitions

display(pd.DataFrame([
    {
        "partition": name,
        "rows": DEVELOPMENT_SPLIT["row_counts"][name],
        "used_for_exploration": name == EDA_PARTITION,
        "membership_sha256": DEVELOPMENT_SPLIT["membership_sha256"][name],
    }
    for name in ("train", "validation", "test")
]))
print(f"Distribution-aware exploration scope: {EDA_PARTITION} ({len(development_df)} rows)")

## 10. Target Class Distribution and Balance (Training Partition)


In [ ]:
from scripts.export_figures import export_figure
from scripts.analyze_target import (
    analyze_target_distribution,
    plot_target_distribution,
)


target_report = analyze_target_distribution(
    development_df,
    target=target_contract.target,
    expected_classes=target_contract.expected_classes,
)

# Revalidate the target contract before interpreting class balance.
target_report.raise_if_invalid()

display(target_report.summary_frame())
display(target_report.distribution_frame(format_percentages=True))

target_distribution_figure = plot_target_distribution(
    target_report,
    title="Dry Bean Target Class Distribution — Training Partition",
)
export_figure(
    target_distribution_figure,
    PROJECT.root / "docs/images/target_class_distribution.png",
)
target_distribution_figure

## 11. Numerical Feature Exploration (Training Partition)


In [ ]:
from IPython.display import display

from scripts.analyze_numerical import (
    analyze_numerical_features,
    plot_numerical_boxplots,
    plot_numerical_distributions,
)


# The validated candidate-feature set is entirely numerical for Dry Bean.
NUMERICAL_FEATURES = FEATURE_COLUMNS

# Explicit exploratory heuristics; neither one changes the raw data.
IQR_MULTIPLIER = 1.5
SKEWNESS_REVIEW_THRESHOLD = 0.5

numerical_report = analyze_numerical_features(
    development_df,
    features=NUMERICAL_FEATURES,
    iqr_multiplier=IQR_MULTIPLIER,
)

numerical_report.raise_if_invalid(
    require_features_present=True,
    require_numeric_conversion=True,
    require_materialization_conditions=True,
    require_no_missing_values=True,
    require_no_outliers=False,
)

display(
    numerical_report.exploration_frame(
        skewness_review_threshold=SKEWNESS_REVIEW_THRESHOLD,
    )
)

distribution_figure = plot_numerical_distributions(
    numerical_report,
    columns=4,
)
display(distribution_figure)

outlier_figure = plot_numerical_boxplots(
    numerical_report,
    columns=4,
)
display(outlier_figure)

if numerical_report.has_outliers:
    print(
        "IQR candidates are descriptive evidence only; no rows are removed "
        "or clipped during exploration."
    )
    display(numerical_report.outlier_summary_frame())

## 12. Feature Relationships and Redundancy (Training Partition)


In [ ]:
from IPython.display import display

from scripts.analyze_relationships import (
    analyze_numerical_feature_relationships,
    plot_numerical_correlation_heatmap,
)


# Exploratory thresholds remain explicit; they flag review candidates only.
STRONG_FEATURE_ASSOCIATION_THRESHOLD = 0.80
REDUNDANCY_REVIEW_THRESHOLD = 0.90

feature_relationship_report = analyze_numerical_feature_relationships(
    development_df,
    features=NUMERICAL_FEATURES,
    strong_association_threshold=STRONG_FEATURE_ASSOCIATION_THRESHOLD,
    redundancy_review_threshold=REDUNDANCY_REVIEW_THRESHOLD,
)

feature_relationship_report.raise_if_invalid(
    require_features_present=True,
    require_sufficient_variation=True,
)

display(feature_relationship_report.numerical_summary_frame())

relationship_review = feature_relationship_report.numerical_review_frame()
if relationship_review.empty:
    print("No feature pairs reached the strong-association review threshold.")
else:
    display(relationship_review)

correlation_figure = plot_numerical_correlation_heatmap(
    feature_relationship_report,
    method="pearson",
    title="Dry Bean Numerical Feature Correlations — Training Partition",
)
export_figure(
    correlation_figure,
    PROJECT.root / "docs/images/numerical_feature_correlation_heatmap.png",
)
display(correlation_figure)

if feature_relationship_report.has_redundancy_candidates:
    print(
        "High association marks redundancy candidates for later review; "
        "no feature is removed during exploration."
    )

## 13. Feature-to-Target Relationships (Training Partition)


In [ ]:
from IPython.display import display

from scripts.analyze_target_relationships import (
    analyze_multiclass_numerical_target_relationships,
    plot_multiclass_feature_target_associations,
)


# Exploratory association strength used for review, not feature selection.
MULTICLASS_ASSOCIATION_REVIEW_THRESHOLD = 0.10

feature_target_report = analyze_multiclass_numerical_target_relationships(
    development_df,
    features=NUMERICAL_FEATURES,
    target=target_contract.target,
    expected_target_classes=target_contract.expected_classes,
    association_review_threshold=MULTICLASS_ASSOCIATION_REVIEW_THRESHOLD,
)

feature_target_report.raise_if_invalid(
    require_features_present=True,
    require_multiclass_target=True,
    require_no_missing_target=True,
    require_expected_target_classes=True,
    require_no_unexpected_target_classes=True,
    require_sufficient_variation=True,
)

display(feature_target_report.summary_frame())
display(feature_target_report.relationships_frame())

association_figure = plot_multiclass_feature_target_associations(
    feature_target_report,
    title="Dry Bean Univariate Feature-to-Target Associations — Training Partition",
)
export_figure(
    association_figure,
    PROJECT.root / "docs/images/feature_target_association_ranking.png",
)
display(association_figure)

if feature_target_report.has_review_candidates:
    print(
        "Review candidates show univariate class association only; no feature "
        "is selected or removed at this stage."
    )

## 14. Class Profiles, Separation, and Overlap (Training Partition)


In [ ]:
from IPython.display import display

from scripts.analyze_class_profiles import (
    analyze_multiclass_class_profiles,
    plot_class_pca_projection,
    plot_standardized_class_profiles,
)


# Presentation choices remain explicit; they do not change the analysis data.
PAIRWISE_OVERLAP_REVIEW_LIMIT = 10
PCA_VISUALIZATION_MAX_POINTS_PER_CLASS = 500
PCA_VISUALIZATION_RANDOM_STATE = 42

class_profile_report = analyze_multiclass_class_profiles(
    development_df,
    features=feature_target_report.requested_features,
    target=target_contract.target,
    expected_target_classes=target_contract.expected_classes,
)

class_profile_report.raise_if_invalid()

display(class_profile_report.summary_frame())
display(
    class_profile_report.pairwise_overlap_frame(
        limit=PAIRWISE_OVERLAP_REVIEW_LIMIT,
    )
)

profile_figure = plot_standardized_class_profiles(
    class_profile_report,
    title="Dry Bean Robust Standardized Class Profiles — Training Partition",
)
export_figure(
    profile_figure,
    PROJECT.root / "docs/images/standardized_class_profiles.png",
)
display(profile_figure)

pca_figure = plot_class_pca_projection(
    class_profile_report,
    max_points_per_class=PCA_VISUALIZATION_MAX_POINTS_PER_CLASS,
    random_state=PCA_VISUALIZATION_RANDOM_STATE,
    title="Dry Bean Exploratory PCA Class Projection — Training Partition",
)
export_figure(
    pca_figure,
    PROJECT.root / "docs/images/class_pca_projection.png",
)
display(pca_figure)

print(
    "Pairwise profile gaps, IQR overlap, and PCA proximity are descriptive "
    "evidence only; they do not estimate classifier error or confusion."
)

## 15. Potential Data Leakage and Derived-Feature Dependencies (Training Partition)

Derived-dependency confirmation drives the later dependency-aware feature-policy ablation, so it is computed on the training partition only.


In [ ]:
from IPython.display import display

from scripts.analyze_leakage import (
    analyze_static_classification_leakage,
    ellipse_eccentricity_dependency,
    equivalent_circle_diameter_dependency,
    external_dependency,
    ratio_dependency,
    roundness_dependency,
    shape_factor_3_dependency,
    shape_factor_4_dependency,
)


# Dataset-specific mathematical dependencies remain explicit in the notebook.
DERIVED_FEATURE_DEPENDENCIES = (
    ratio_dependency(
        "AspectRatio",
        numerator="MajorAxisLength",
        denominator="MinorAxisLength",
    ),
    ellipse_eccentricity_dependency(
        "Eccentricity",
        major_axis="MajorAxisLength",
        minor_axis="MinorAxisLength",
    ),
    equivalent_circle_diameter_dependency("EquivDiameter", area="Area"),
    external_dependency(
        "Extent",
        sources=("Area", "BoundingBoxArea"),
        formula="Area / BoundingBoxArea",
        note="BoundingBoxArea is required by the source definition but is not retained.",
    ),
    ratio_dependency("Solidity", numerator="Area", denominator="ConvexArea"),
    roundness_dependency("Roundness", area="Area", perimeter="Perimeter"),
    ratio_dependency(
        "Compactness",
        numerator="EquivDiameter",
        denominator="MajorAxisLength",
    ),
    ratio_dependency(
        "ShapeFactor1",
        numerator="MajorAxisLength",
        denominator="Area",
    ),
    ratio_dependency(
        "ShapeFactor2",
        numerator="MinorAxisLength",
        denominator="Area",
    ),
    shape_factor_3_dependency(
        "ShapeFactor3",
        area="Area",
        major_axis="MajorAxisLength",
    ),
    shape_factor_4_dependency(
        "ShapeFactor4",
        area="Area",
        major_axis="MajorAxisLength",
        minor_axis="MinorAxisLength",
    ),
)

# Tolerances account for rounding in released derived measurements.
DERIVED_DEPENDENCY_RELATIVE_TOLERANCE = 1e-4
DERIVED_DEPENDENCY_ABSOLUTE_TOLERANCE = 1e-9
DERIVED_DEPENDENCY_CONFIRMATION_RATE = 0.99

leakage_report = analyze_static_classification_leakage(
    development_df,
    target=target_contract.target,
    candidate_features=FEATURE_COLUMNS,
    identifiers=IDENTIFIER_COLUMNS,
    derived_dependencies=DERIVED_FEATURE_DEPENDENCIES,
    relative_tolerance=DERIVED_DEPENDENCY_RELATIVE_TOLERANCE,
    absolute_tolerance=DERIVED_DEPENDENCY_ABSOLUTE_TOLERANCE,
    confirmation_rate=DERIVED_DEPENDENCY_CONFIRMATION_RATE,
)

leakage_report.raise_if_invalid()

display(leakage_report.summary_frame())
display(leakage_report.dependency_frame())

if leakage_report.has_target_proxy_candidates:
    print("Potential direct target proxies require review:")
    display(leakage_report.target_proxy_candidates_frame())
else:
    print("No direct or deterministic target proxy was detected among candidate features.")

print(
    "Confirmed mathematical dependencies indicate structural redundancy, not target leakage. "
    "No feature is removed during exploration."
)

## 16. Initial Data-Quality Findings

Structural findings (domain, value, and duplicate checks) cover the complete source; distribution-dependent findings (class support, IQR candidates, dependencies) cover the training partition only.


In [ ]:
from scripts.consolidate_quality_findings import (
    consolidate_initial_data_quality_from_reports,
)


quality_findings_report = consolidate_initial_data_quality_from_reports(
    available_fields=df.columns,
    # Structural source size; distribution-aware inputs are training-only.
    row_count=len(df),
    domain_report=domain_report,
    value_quality_report=value_quality_report,
    duplicate_report=duplicate_report,
    target_report=target_report,
    numerical_report=numerical_report,
    leakage_report=leakage_report,
)

# Preparation actions are intentionally defined later, in stage 18.
quality_findings_report.raise_if_invalid(
    require_action_for_open_findings=False,
)

display(quality_findings_report.quality_overview_frame())

initial_quality_findings = quality_findings_report.findings_frame()
if initial_quality_findings.empty:
    print("No initial data-quality finding requires review or resolution.")
else:
    display(initial_quality_findings)
    display(quality_findings_report.evidence_frame())

display(quality_findings_report.validated_non_issues_frame())

## 17. Key Exploratory Insights


In [ ]:
from scripts.consolidate_exploratory_insights import (
    consolidate_key_exploratory_insights_from_reports,
)


exploratory_insights_report = consolidate_key_exploratory_insights_from_reports(
    available_fields=df.columns,
    quality_report=quality_findings_report,
    target_report=target_report,
    numerical_report=numerical_report,
    feature_relationship_report=feature_relationship_report,
    feature_target_report=feature_target_report,
    class_profile_report=class_profile_report,
    leakage_report=leakage_report,
)

exploratory_insights_report.raise_if_invalid()

display(exploratory_insights_report.exploratory_overview_frame())
display(exploratory_insights_report.key_insights_frame())

exploratory_hypotheses = exploratory_insights_report.hypotheses_frame()
if not exploratory_hypotheses.empty:
    print(
        "The following hypotheses remain exploratory and must be validated "
        "in later modeling stages."
    )
    display(exploratory_hypotheses)

display(exploratory_insights_report.limitations_frame())

## 18. Preparation Decisions

The split policy recorded here is the one already frozen in section 9b; the remaining decisions are derived from structural checks and training-only exploration.


In [ ]:
from IPython.display import display

from scripts.record_preparation_decisions import (
    record_static_multiclass_preparation_decisions,
)


# The split policy is the one frozen in section 9b before exploration.
preparation_decision_report = record_static_multiclass_preparation_decisions(
    available_fields=df.columns,
    target=target_contract.target,
    target_classes=target_contract.expected_classes,
    candidate_features=FEATURE_COLUMNS,
    identifiers=IDENTIFIER_COLUMNS,
    duplicate_report=duplicate_report,
    target_report=target_report,
    numerical_report=numerical_report,
    feature_relationship_report=feature_relationship_report,
    leakage_report=leakage_report,
    quality_report=quality_findings_report,
    exploratory_insights_report=exploratory_insights_report,
    train_fraction=DEVELOPMENT_SPLIT_POLICY.train_fraction,
    validation_fraction=DEVELOPMENT_SPLIT_POLICY.validation_fraction,
    test_fraction=DEVELOPMENT_SPLIT_POLICY.test_fraction,
    random_seed=DEVELOPMENT_SPLIT_POLICY.random_seed,
)

preparation_decision_report.raise_if_invalid()
preparation_decision_report.raise_if_split_not_ready()

display(preparation_decision_report.summary_frame())
display(preparation_decision_report.decisions_frame())
display(preparation_decision_report.execution_plan_frame())
display(preparation_decision_report.split_policy_frame())
display(preparation_decision_report.guardrails_frame())

preparation_blockers = preparation_decision_report.blockers_frame()
if preparation_blockers.empty:
    print("No preparation or split-execution blocker remains open.")
else:
    display(preparation_blockers)

print(
    "Preparation decisions are declarations only; apart from the section-9b "
    "split freeze, no cleaning, transformation, resampling, feature selection, "
    "or model fitting was executed."
)

## 19. Handoff and Next Steps


In [ ]:
from IPython.display import display

from scripts.build_exploration_handoff import (
    build_static_multiclass_exploration_handoff,
)


EXPLORATION_HANDOFF_FILE = PROJECT.path(
    "artifacts",
    "exploration",
    DATASET_SLUG,
    "exploration-handoff.json",
)

exploration_handoff_report = build_static_multiclass_exploration_handoff(
    dataset_slug=DATASET_SLUG,
    source_repository="UCI Machine Learning Repository",
    source_dataset_id=UCI_DATASET_ID,
    source_file=DATASET_FILE,
    project_root=PROJECT.root,
    source_dataframe=df,
    target_contract=target_contract,
    feature_columns=FEATURE_COLUMNS,
    numerical_features=NUMERICAL_FEATURES,
    identifier_columns=IDENTIFIER_COLUMNS,
    target_report=target_report,
    duplicate_report=duplicate_report,
    feature_relationship_report=feature_relationship_report,
    leakage_report=leakage_report,
    quality_report=quality_findings_report,
    insights_report=exploratory_insights_report,
    preparation_report=preparation_decision_report,
    class_profile_report=class_profile_report,
    development_split=DEVELOPMENT_SPLIT,
)

exploration_handoff_report.raise_if_invalid()
persisted_handoff = exploration_handoff_report.write(EXPLORATION_HANDOFF_FILE)

display(exploration_handoff_report.summary_frame())

open_reviews = exploration_handoff_report.open_reviews_frame()
if not open_reviews.empty:
    print("Non-blocking reviews carried forward:")
    display(open_reviews)

display(exploration_handoff_report.next_steps_frame())
display(exploration_handoff_report.expected_outputs_frame())

print(f"Persisted handoff: {PROJECT.display(persisted_handoff.path)}")
print(f"SHA-256: {persisted_handoff.sha256}")
print(
    "Notebook 01 is complete. Notebook 02 must reload and revalidate the "
    "source from a fresh kernel and reproduce the split frozen in section 9b "
    "before materializing preparation artifacts."
)